In [ ]:
import requests
import pandas as pd

In [ ]:
API_KEY = "vfkQQ0xCxtZG6pRYnhmTeACfZw1xsm63AXY8N3XM"
SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"
NUTRIENT_MAP = {
    "protein": "Protein",
    "calories": "Energy",
    "fiber": "Fiber, total dietary",
    "sodium": "Sodium, Na",
    "fat": "Total lipid (fat)",
    "carbs": "Carbohydrate, by difference"
}

In [ ]:
def get_nutrient_value(food, nutrient_name):
    food_nutrients = food.get("foodNutrients", [])
    for nutrient in food_nutrients:
        if nutrient.get("nutrientName") == nutrient_name:
            return nutrient.get("value", None)
    return None

In [ ]:
def search_specific_food(input):
  parameters = {"query": input,"api_key": API_KEY}
  response = requests.get('https://api.nal.usda.gov/fdc/v1/foods/search',params = parameters)
  response = response.json()
  data_count = 0
  for food in response.get('foods', []):
        if food.get('dataType') == 'Branded':
            continue
        print(f"{data_count+1})")
        print("This food is a",food.get('dataType'))
        print(food.get('description'))
        #print("Score: ",food.get('score'))

        if food.get('ingredients') is not None:
          print("Ingredients: ", food.get('ingredients'))

        for nutrient in food.get('foodNutrients', []):

            if nutrient.get('nutrientName')in NUTRIENT_MAP.values():
              if food.get('dataType') == 'SR Legacy' and nutrient.get('nutrientName') == 'Energy' and nutrient.get('unitName') == 'kJ':
                    continue
              print("Name:",nutrient.get('nutrientName'), "  Value: ",nutrient.get('value')," "+nutrient.get('unitName'))



        print()
        data_count+=1
        if data_count >=8:
          break

search_specific_food('banana')


1)
This food is a SR Legacy
Bananas, dehydrated, or banana powder
Name: Protein   Value:  3.89  G
Name: Sodium, Na   Value:  3.0  MG
Name: Total lipid (fat)   Value:  1.81  G
Name: Carbohydrate, by difference   Value:  88.3  G
Name: Energy   Value:  346  KCAL
Name: Fiber, total dietary   Value:  9.9  G

2)
This food is a Survey (FNDDS)
Banana, baked
Name: Protein   Value:  0.82  G
Name: Total lipid (fat)   Value:  3.21  G
Name: Carbohydrate, by difference   Value:  32.43  G
Name: Energy   Value:  161  KCAL
Name: Fiber, total dietary   Value:  1.8  G
Name: Sodium, Na   Value:  3  MG

3)
This food is a Survey (FNDDS)
Banana, raw
Name: Protein   Value:  0.74  G
Name: Total lipid (fat)   Value:  0.28  G
Name: Carbohydrate, by difference   Value:  22.71  G
Name: Energy   Value:  97  KCAL
Name: Fiber, total dietary   Value:  1.7  G
Name: Sodium, Na   Value:  0  MG

4)
This food is a Survey (FNDDS)
Banana chips
Name: Protein   Value:  2.3  G
Name: Total lipid (fat)   Value:  33.6  G
Name: Car

In [ ]:
def get_food(input):
    p={"query":input,"api_key":API_KEY}
    response =requests.get("https://api.nal.usda.gov/fdc/v1/foods/search", params=p)
    response =response.json()
    return response.get("foods",[])

In [ ]:
def rank_foods(food_name,nutrient_key, topn=5, highest=True):
    if nutrient_key not in NUTRIENT_MAP:
        print("Sorry, we do not have this information!.")
        return
    nutrient=NUTRIENT_MAP[nutrient_key]
    foods = get_food(food_name)
    ranked_list =[]
    for food in foods:
        if food.get("dataType") =="Branded":
            continue
        description = food.get("description","Unknown Food")
        data_type = food.get("dataType","Unknown Type")
        nutrient_value = get_nutrient_value(food,nutrient)
        if nutrient_value is not None:
            ranked_list.append({"Food": description,"Type": data_type, nutrient_key.capitalize(): nutrient_value})
    ranked_list = sorted(ranked_list,key=lambda x: x[nutrient_key.capitalize()],reverse=highest)
    df =pd.DataFrame(ranked_list[:topn])
    return df
rank_foods("pasta", "fiber", topn=5, highest=True)

,Food,Type,Fiber
0,"Pasta, vegetable, cooked",Survey (FNDDS),4.3
1,"Pasta, whole grain, cooked",Survey (FNDDS),3.9
2,"Pasta, dry, enriched",SR Legacy,3.2
3,"Pasta, dry, unenriched",SR Legacy,3.2
4,"Pasta with sauce, meatless, school lunch",Survey (FNDDS),2.9


In [1]:
from collections import Counter
import math
#types of scores
intent_corpus = {
    "lookup": "how many calories in protein in amount of vitamin c in",
    "comparison": "is higher than more protein than compare vs versus healthier",
    "ranking": "highest top best most lowest least richest source of",
    "educational": "what is why do we need explain define importance of"
}

#calculate the scores to categorize the query - match based on total score
def scorecalcjm(query, doc_text, collection_text, lambd=0.5):
    queryTerms =query.lower().split()
    docTerms = doc_text.lower().split()
    collectionTerms = collection_text.lower().split()
    doc_count =Counter(docTerms)
    coll_count = Counter(collectionTerms)
    doc_len = len(docTerms)
    coll_len =len(collectionTerms)

    log_likelihood= 0.0
    vocab_size =len(set(collectionTerms))

    for word in queryTerms:
        # Ensure we are using the correct count variables (doc_count, coll_count)
        pwd = doc_count[word] /doc_len if doc_len > 0 else 0
        pwc = (coll_count[word] + 0.1)/ (coll_len + (0.1 * vocab_size))
        # smoothing for zero
        smooth = (lambd*pwd)+((1 - lambd) * pwc)
        # make sure value greater than z for no crash
        log_likelihood+= math.log(smooth)
    return log_likelihood

# testing
all_text = " ".join(intent_corpus.values())
user_query = "is beef higher in protein than chicken"
scores = {intent: scorecalcjm(user_query, text, all_text) for intent, text in intent_corpus.items()}
predicted_intent = max(scores, key=scores.get)

print(f"Scores: {scores}")
print(f"Predicted Intent: {predicted_intent}")
all_text = " ".join(intent_corpus.values())
user_query = "what has the most calories"
scores = {intent: scorecalcjm(user_query, text, all_text) for intent, text in intent_corpus.items()}
predicted_intent = max(scores, key=scores.get)
print(f"Scores: {scores}")
print(f"Predicted Intent: {predicted_intent}")

Scores: {'lookup': -29.755311086993895, 'comparison': -26.91314301829718, 'ranking': -32.38174456948326, 'educational': -31.26270737596311}
Predicted Intent: comparison
Scores: {'lookup': -25.104364583266523, 'comparison': -26.62573872441707, 'ranking': -24.94421508521001, 'educational': -25.029109770275547}
Predicted Intent: ranking
